# RAG Pipeline - UK Standard Visitor Visa


This notebook builds a beginner-friendly RAG pipeline for the **UK Standard Visitor visa** PDF.

RAG means **Retrieval-Augmented Generation**. That sounds big, but the idea is simple:

1. Load a document.
2. Break the document into smaller pieces.
3. Convert those pieces into numbers called embeddings.
4. Store those embeddings in a searchable vector database.
5. When a user asks a question, retrieve the most relevant pieces.
6. Give only those pieces to a language model so it answers from the document.

We are starting with free/local Hugging Face models because paid hosted model services cost money. At the beginning of a project, it is better to learn the moving parts first before paying for stronger models.

By the end of this notebook, you will understand the full RAG path from PDF to answer.


## Step 1 - Install Libraries


Notes:

This cell installs the main libraries used in the project.

- `langchain` gives us useful building blocks for RAG pipelines.
- `langchain-community` contains community integrations such as PDF loaders, embeddings, and FAISS vector stores.
- `pypdf` lets Python read text from PDF files.
- `sentence-transformers` gives us a free embedding model for turning text into vectors.
- `faiss-cpu` stores vectors and searches them quickly.
- `transformers` lets us load the free Hugging Face generation model.
- `accelerate` helps Hugging Face models run smoothly on your machine.

For a first RAG project, think of these as the toolbox. We need one tool for loading documents, one for splitting text, one for embeddings, one for search, and one for generating answers.


In [ ]:
!pip install langchain==0.2.16
!pip install langchain-community==0.2.16
!pip install langchain-core==0.2.38
!pip install faiss-cpu==1.13.2
!pip install pypdf==4.3.1
!pip install sentence-transformers==3.0.1
!pip install transformers accelerate

## Step 2 - Check Package Versions


Notes:

Version checks are useful because AI libraries change quickly. A notebook that worked last month can fail if a library changes a class name, import path, or function behavior.

This cell prints the installed versions so you can debug problems more easily. If you ever ask for help, sharing these versions makes the issue much easier to reproduce.

Beginner tip: when something fails after installation, check versions before changing code. Many RAG errors are version mismatch errors, not logic errors.


In [ ]:
import importlib.metadata as metadata

packages = [
    "langchain",
    "langchain-core",
    "langchain-community",
    "faiss-cpu",
    "pypdf",
    "sentence-transformers",
    "transformers",
    "accelerate",
]

for package in packages:
    print(f"{package}: {metadata.version(package)}")

## Step 3 - Import Libraries


Notes:

This cell imports everything we will use later.

Each import belongs to one part of the RAG pipeline:

- `re` is used to clean messy PDF text with regular expressions.
- `Path` helps us build file paths that work across folders.
- `PyPDFLoader` reads the PDF and turns each page into a LangChain document.
- `RecursiveCharacterTextSplitter` breaks long text into smaller overlapping chunks.
- `HuggingFaceEmbeddings` creates vector embeddings from text.
- `FAISS` stores and searches those vectors.
- `pipeline` loads the free Hugging Face text generation model.

At this stage, we are only preparing the ingredients. The actual pipeline starts when we load the PDF.


In [ ]:
import re

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

## Step 4 - Load the PDF


Notes:

This is the document-loading step.

A PDF is not automatically useful to a language model. First, we need to extract the text from it. `PyPDFLoader` reads the PDF and returns a list of `Document` objects.

A LangChain `Document` usually has two important parts:

- `page_content`: the actual text extracted from the page.
- `metadata`: extra information such as the file name and page number.

The notebook looks for `Data/Standard-Visitor-Route.pdf` first, because your PDFs are now stored in the `Data` folder.


In [ ]:
from pathlib import Path

candidate_paths = [
    Path("Data/Standard-Visitor-Route.pdf"),
    Path("RAG-Experiments/Data/Standard-Visitor-Route.pdf"),
]

PDF_PATH = next((path for path in candidate_paths if path.exists()), None)

if PDF_PATH is None:
    raise FileNotFoundError("Could not find Standard-Visitor-Route.pdf inside the Data folder.")

loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()

print(f"Loaded {len(documents)} pages from {PDF_PATH}")
print(documents[0].metadata)


## Step 5 - Inspect Loaded Text


Notes:

Before cleaning or chunking, always inspect the raw text.

PDF extraction is often messy. You may see page numbers, headers, footers, repeated website text, broken spacing, or words joined together. This is normal.

This cell prints the first part of the first page so you can see what the loader extracted. A good RAG engineer checks the raw text early because bad source text usually leads to bad retrieval later.


In [ ]:
print(documents[0].page_content[:1200])
print("\nMetadata:", documents[0].metadata)

## Step 6 - Clean PDF Text


Notes:

This step removes some repeated noise from the PDF text.

Cleaning matters because embeddings are created from the text we provide. If the chunks contain lots of repeated footer text or web links, the retriever may match on noise instead of useful visa information.

This cleaning function does three simple beginner-friendly things:

- replaces line breaks with spaces,
- removes repeated GOV.UK print footer patterns,
- collapses extra spaces into one clean space.

The goal is not perfect text. The goal is cleaner text that is easier to search.


In [ ]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\d{2}/\d{2}/\d{4},\s*\d{2}:\d{2}\s*Print.*?GOV\.UK", " ", text)
    text = re.sub(r"https://www\.gov\.uk/[^\s]+\s+\d+/\d+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for doc in documents:
    doc.page_content = clean_text(doc.page_content)

print(documents[0].page_content[:800])

## Step 7 - Split Text Into Chunks


Notes:

Language models and embedding models work better with smaller pieces of text.

If we embed the whole PDF as one giant block, retrieval becomes too broad. If the chunks are too tiny, each chunk may lose important context. Chunking is about finding a practical middle ground.

In this notebook:

- `chunk_size=500` means each chunk is roughly 500 characters.
- `chunk_overlap=50` means nearby chunks share a little text.

Overlap is useful because important sentences often sit on the boundary between two chunks. The overlap reduces the chance that we cut the meaning in half.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

docs = text_splitter.split_documents(documents)
print(f"Created {len(docs)} chunks")

## Step 8 - Inspect Chunks


Notes:

This cell prints a few chunks so you can check whether chunking worked.

You are looking for chunks that are:

- readable,
- not too long,
- not too short,
- connected enough to make sense on their own.

If chunks look messy here, fix cleaning or chunk settings before moving on. Retrieval quality depends heavily on chunk quality.


In [ ]:
for i, doc in enumerate(docs[:5], start=1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content[:500])
    print("Metadata:", doc.metadata)

## Step 9 - Create Embeddings


Notes:

Embeddings are the bridge between text and search.

A computer cannot directly compare meanings the way humans do. An embedding model converts text into a list of numbers called a vector. Texts with similar meanings should have vectors that are close together.

Example:

- "extend visitor visa"
- "stay longer as a visitor"

These phrases use different words, but their meanings are related. Embeddings help the retriever understand that relationship.

We use `sentence-transformers/all-MiniLM-L6-v2` because it is free, fast, and good enough for learning RAG.


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## Step 10 - Store Chunks In FAISS


Notes:

FAISS is the vector database for this notebook.

After we create embeddings for all chunks, we need somewhere to store and search them. FAISS keeps the chunk vectors and lets us quickly find the chunks closest to a question vector.

This is the first point where the RAG system becomes searchable. Before this step, we only had documents and chunks. After this step, we have a knowledge base that can retrieve relevant evidence.


In [ ]:
vectorstore = FAISS.from_documents(
    docs,
    embeddings,
)

## Step 11 - Create Retriever


Notes:

A retriever is a simple search interface over the vector store.

When the user asks a question, the retriever:

1. embeds the question,
2. compares the question vector with stored chunk vectors,
3. returns the closest matching chunks.

The setting `k=5` means "return the top 5 most relevant chunks." A higher value gives the model more context, but too much context can confuse a small local model.


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

## Step 12 - Retrieve Evidence


Notes:

This cell tests retrieval before generation.

That is an important habit. If the retrieved chunks are wrong, the final answer will probably be wrong too. RAG has two main jobs:

- retrieval: find the right evidence,
- generation: turn that evidence into a readable answer.

Here we only test retrieval. We ask a question and inspect the chunks that FAISS returns.


In [ ]:
query = "Can I extend my stay as a Standard Visitor?"

retrieved_docs = retriever.invoke(query)

def display_retrieved_docs(results):
    for i, doc in enumerate(results, start=1):
        print(f"\n{'=' * 60}")
        print(f"Retrieved Chunk {i}")
        print(f"{'=' * 60}")
        print("Source:", doc.metadata)
        print("Content:")
        print(doc.page_content)

display_retrieved_docs(retrieved_docs)

## Step 13 - Load Free Local Generation Model


Notes:

This step loads the language model that writes the final answer.

We are using `google/flan-t5-base` through Hugging Face. It is free and works well for learning, but it is much smaller than paid hosted models.

That means:

- it may give shorter answers,
- it may miss nuance,
- it needs concise context,
- it performs best when the prompt is very clear.

For a first RAG project, this is a good tradeoff. You learn the architecture without paying for model calls.


In [ ]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
)

## Step 14 - Build Context


Notes:

The retrieved chunks are now combined into one context block.

This context is the evidence we will give to the generation model. The model should answer from this context, not from memory.

The context is capped because small local models have limited input capacity. If we send too much text, the model may become slow, truncate the prompt, or produce weaker answers.

Beginner rule: give the model enough evidence to answer, but not so much that the important information gets buried.


In [ ]:
context = "\n\n".join(
    doc.page_content for doc in retrieved_docs[:3]
)[:1800]

print(context)

## Step 15 - Create Grounded Prompt


Notes:

The prompt is where we tell the model how to behave.

A grounded prompt usually contains:

- an instruction,
- the retrieved context,
- the user's question,
- a clear place for the answer.

The key instruction here is that the model should answer using only the provided context. This helps reduce hallucination because the model is guided back to the document evidence.


In [ ]:
prompt = f"""
Answer the question using only the context below.
If the answer is not in the context, say: I could not find the answer in the provided document.

Context:
{context}

Question: {query}

Answer:
"""

print(prompt[:1200])

## Step 16 - Generate Answer


Notes:

This is the final RAG step.

The model receives the grounded prompt and generates an answer. At this point, the answer is based on the retrieved PDF chunks rather than only on the model's built-in knowledge.

When you evaluate the answer, check two things:

1. Did retrieval find the right evidence?
2. Did generation explain that evidence correctly?

If the answer is weak, debug in that order. First inspect retrieval. Then improve the prompt or try a stronger model later when you are ready.


In [ ]:
response = generator(
    prompt,
    max_new_tokens=160,
    do_sample=False,
)

print(response[0]["generated_text"])